In [1]:
%pip install -q -U pandas requests tqdm
from datetime import datetime, timezone
import time
import pandas as pd
import requests
from tqdm.auto import tqdm

NETID = 'jvijayak'
PROJECTS = {
    'luispedro_mahotas': 'luispedro/mahotas',
    'volkamerlab_opencadd': 'volkamerlab/opencadd',
    'ecshackweek_impedance.py': 'ECSHackWeek/impedance.py',
    'cesiumgs_3d-tiles': 'CesiumGS/3d-tiles',
    'kbarbary_sep': 'kbarbary/sep',
    'cannylab_tsne-cuda': 'CannyLab/tsne-cuda',
    'jmchandonia_coral': 'jmchandonia/CORAL',
    'mouseland_rastermap': 'MouseLand/rastermap',
    'sentinel-hub_sentinelhub-py': 'sentinel-hub/sentinelhub-py',
    'grimbough_rhdf5lib': 'grimbough/Rhdf5lib',
}
BATCH_SIZE = 10  # below the assignment's limit of 50
WAIT_SECONDS = 1
WOC_API = 'https://worldofcode.org/api'

def woc_get(path, params=None, retries=4):
    for attempt in range(retries):
        response = requests.get(f'{WOC_API}{path}', params=params, timeout=90)
        if response.status_code == 429 and attempt < retries - 1:
            time.sleep(int(response.headers.get('Retry-After', 10)))
            continue
        response.raise_for_status()
        return response.json()


In [2]:
github_rows = []
for project_wocid, repo in PROJECTS.items():
    response = requests.get(f'https://api.github.com/repos/{repo}', headers={'Accept': 'application/vnd.github+json'}, timeout=30)
    response.raise_for_status()
    data = response.json()
    latest = requests.get(f'https://api.github.com/repos/{repo}/commits', params={'per_page': 1}, headers={'Accept': 'application/vnd.github+json'}, timeout=30)
    latest.raise_for_status()
    github_rows.append({
        'project_wocid': project_wocid,
        'github_repository': repo,
        'stars': data['stargazers_count'],
        'forks': data['forks_count'],
        'last_commit_date': latest.json()[0]['commit']['committer']['date'][:10],
    })
github_df = pd.DataFrame(github_rows)
display(github_df)

,project_wocid,github_repository,stars,forks,last_commit_date
0,luispedro_mahotas,luispedro/mahotas,890,150,2026-09-23
1,volkamerlab_opencadd,volkamerlab/opencadd,111,26,2025-04-16
2,ecshackweek_impedance.py,ECSHackWeek/impedance.py,299,124,2026-04-10
3,cesiumgs_3d-tiles,CesiumGS/3d-tiles,2614,504,2026-09-30
4,kbarbary_sep,kbarbary/sep,210,68,2025-02-18
5,cannylab_tsne-cuda,CannyLab/tsne-cuda,1958,138,2026-07-22
6,jmchandonia_coral,jmchandonia/CORAL,8,3,2026-10-01
7,mouseland_rastermap,MouseLand/rastermap,257,56,2025-04-04
8,sentinel-hub_sentinelhub-py,sentinel-hub/sentinelhub-py,909,255,2026-09-16
9,grimbough_rhdf5lib,grimbough/Rhdf5lib,7,14,2026-06-16


In [3]:
project_commits = {}
for project_wocid in tqdm(PROJECTS, desc='WoC project-to-commit lookups'):
    project_commits[project_wocid] = woc_get(f'/lookup/map/p2c/{project_wocid}')['data']
    time.sleep(WAIT_SECONDS)

sha_df = pd.DataFrame(
    [(project, sha) for project, shas in project_commits.items() for sha in shas],
    columns=['project_wocid', 'commit_sha1'],
)
sha_df.to_csv(f'{NETID}_commit_sha1s.csv', index=False, sep=';')
display(sha_df.groupby('project_wocid').size().rename('number_of_commits').to_frame())

WoC project-to-commit lookups:   0%|          | 0/10 [00:00<?, ?it/s]

,number_of_commits
project_wocid,
cannylab_tsne-cuda,486
cesiumgs_3d-tiles,2579
ecshackweek_impedance.py,1007
grimbough_rhdf5lib,492
jmchandonia_coral,4184
kbarbary_sep,452
luispedro_mahotas,1554
mouseland_rastermap,389
sentinel-hub_sentinelhub-py,2327


In [4]:
rows = []
errors = []
for project_wocid, shas in project_commits.items():
    for start in tqdm(range(0, len(shas), BATCH_SIZE), desc=project_wocid):
        batch = shas[start:start + BATCH_SIZE]
        response = woc_get('/lookup/map/commit.tch', params=[('q', sha) for sha in batch])
        result = response['data']
        batch_errors = response.get('errors', {})
        errors.extend((project_wocid, sha, error) for sha, error in batch_errors.items())
        for sha, values in result.items():
            commit = values[0]
            rows.append({
                'project_wocid': project_wocid,
                'commit_sha1': sha,
                'author': commit[2][0],
                'time': int(commit[2][1]),
                'commit message': commit[4],
            })
        time.sleep(WAIT_SECONDS)

luispedro_mahotas:   0%|          | 0/156 [00:00<?, ?it/s]

volkamerlab_opencadd:   0%|          | 0/142 [00:00<?, ?it/s]

ecshackweek_impedance.py:   0%|          | 0/101 [00:00<?, ?it/s]

cesiumgs_3d-tiles:   0%|          | 0/258 [00:00<?, ?it/s]

kbarbary_sep:   0%|          | 0/46 [00:00<?, ?it/s]

cannylab_tsne-cuda:   0%|          | 0/49 [00:00<?, ?it/s]

jmchandonia_coral:   0%|          | 0/419 [00:00<?, ?it/s]

mouseland_rastermap:   0%|          | 0/39 [00:00<?, ?it/s]

sentinel-hub_sentinelhub-py:   0%|          | 0/233 [00:00<?, ?it/s]

grimbough_rhdf5lib:   0%|          | 0/50 [00:00<?, ?it/s]

In [5]:
summary_df = pd.DataFrame(rows, columns=['project_wocid', 'commit_sha1', 'author', 'time', 'commit message'])
summary_path = f'{NETID}_project_summary.csv'
summary_df.to_csv(summary_path, index=False, sep=';')

assert list(summary_df.columns) == ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
assert not summary_df.empty, 'No commit rows were retrieved.'
assert summary_df['project_wocid'].nunique() == len(PROJECTS), 'At least one assigned project has no rows.'
print(f'Wrote {len(summary_df):,} rows to {summary_path}')
display(summary_df.head())

Wrote 14,821 rows to jvijayak_project_summary.csv


,project_wocid,commit_sha1,author,time,commit message
0,luispedro_mahotas,001745635622c724ac6c4c7188411125677cddeb,Luis Pedro Coelho <luis@luispedro.org>,1416139076,DOC Document PIL\n
1,luispedro_mahotas,0017fc16b9aa51a082e251cc3cabe3f732785c2c,Sujoy Kumar Goswami <sujoykumargoswami@gmail.com>,1591988410,Update edge.py\n\nupdate threshold 't'
2,luispedro_mahotas,004d1d857ee080a0cb453b50865e1db134212636,Luis Pedro Coelho <lpc@cmu.edu>,1281471217,Faster watershed.\n\nThis uses flat indices in...
3,luispedro_mahotas,00933e9879dfc9c905489639a6fdef01ce838766,Luis Pedro Coelho <luis@luispedro.org>,1308935012,DOC Document previous changes\n
4,luispedro_mahotas,00b8c9748e3fc01d794847c67b30d49a7175833c,Luis Pedro Coelho <luis@luispedro.org>,1721242170,DOC Fix README badges\n


In [6]:
woc_stats = (summary_df.groupby('project_wocid', as_index=False)
    .agg(number_of_commits=('commit_sha1', 'size'),
         number_of_authors=('author', 'nunique'),
         max_time=('time', 'max'),
         min_time=('time', 'min')))
woc_stats['max_time_utc'] = pd.to_datetime(woc_stats['max_time'], unit='s', utc=True)
woc_stats['min_time_utc'] = pd.to_datetime(woc_stats['min_time'], unit='s', utc=True)
part1_report = woc_stats.merge(github_df, on='project_wocid', how='left')
part1_report.to_csv(f'{NETID}_part1_metadata.csv', index=False, sep=';')
display(part1_report)

,project_wocid,number_of_commits,number_of_authors,max_time,min_time,max_time_utc,min_time_utc,github_repository,stars,forks,last_commit_date
0,cannylab_tsne-cuda,485,11,1732457883,1521854093,2024-11-24 14:18:03+00:00,2018-03-24 01:14:53+00:00,CannyLab/tsne-cuda,1958,138,2026-07-22
1,cesiumgs_3d-tiles,2579,64,1757278627,1433503403,2025-09-07 20:57:07+00:00,2015-06-05 11:23:23+00:00,CesiumGS/3d-tiles,2614,504,2026-09-30
2,ecshackweek_impedance.py,1006,60,1776216561,1526508236,2026-04-15 01:29:21+00:00,2018-05-16 22:03:56+00:00,ECSHackWeek/impedance.py,299,124,2026-04-10
3,grimbough_rhdf5lib,492,24,1762047875,1499419330,2025-11-02 01:44:35+00:00,2017-07-07 09:22:10+00:00,grimbough/Rhdf5lib,7,14,2026-06-16
4,jmchandonia_coral,4184,9,1775861165,1528937748,2026-04-10 22:46:05+00:00,2018-06-14 00:55:48+00:00,jmchandonia/CORAL,8,3,2026-10-01
5,kbarbary_sep,452,34,1770115011,1394735767,2026-02-03 10:36:51+00:00,2014-03-13 18:36:07+00:00,kbarbary/sep,210,68,2025-02-18
6,luispedro_mahotas,1497,50,1755596152,1218575941,2025-08-19 09:35:52+00:00,2008-08-12 21:19:01+00:00,luispedro/mahotas,890,150,2026-09-23
7,mouseland_rastermap,389,13,1743788307,1528984563,2025-04-04 17:38:27+00:00,2018-06-14 13:56:03+00:00,MouseLand/rastermap,257,56,2025-04-04
8,sentinel-hub_sentinelhub-py,2326,74,1759849762,1495023000,2025-10-07 15:09:22+00:00,2017-05-17 12:10:00+00:00,sentinel-hub/sentinelhub-py,909,255,2026-09-16
9,volkamerlab_opencadd,1411,29,1759424604,1580728313,2025-10-02 17:03:24+00:00,2020-02-03 11:11:53+00:00,volkamerlab/opencadd,111,26,2025-04-16
